# 04 — Export Final Model

Build an `EModelExportFinalModelScanConfig`, expand it via `GridScanGenerationTask`,
and run the `EModelExportFinalModelTask` for each coordinate. The task seeds its
working directory from the analysis output, then calls `export_emodels_hoc` and/or
`export_emodels_sonata` with the seeds the user picked.

**Reads from:** `obi-output/03_analysis_and_validation/grid_scan/0/`.

**Writes to:** `obi-output/04_export_final_model/grid_scan/0/` — the working directory
plus the new `export_emodels_hoc/` (one `.hoc` per validated model) and
`export_emodels_sonata/` (with `nodes.h5`).


## Imports

In [ ]:
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.emodel_optimization._04_export_final_model.blocks import (
    ExportInitialize,
    ExportSettings,
)


## Build the scan config

In [ ]:
previous_stage = (
    Path.cwd() / "../../../obi-output/03_analysis_and_validation/grid_scan/0"
).resolve()
assert previous_stage.exists(), (
    f"{previous_stage} not found — run 03_analysis_and_validation.ipynb first."
)

scan_config = obi.EModelExportFinalModelScanConfig(
    initialize=ExportInitialize(
        previous_stage_output_path=previous_stage,
        emodel="L5PC",
        species="rat",
        brain_region="SSCX",
    ),
    export_settings=ExportSettings(
        export_hoc=True,
        export_sonata=True,
        only_validated=False,
        only_best=False,
        seeds=(1,),  # must be a subset of the seeds run in stage 02
    ),
)


## Run the grid scan

In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/04_export_final_model/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)


## Inspect the exported model

In [ ]:
coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print("Coordinate output:", coord_root)

for tree in ("export_emodels_hoc", "export_emodels_sonata"):
    print(f"\n{tree}/")
    for p in sorted((coord_root / tree).rglob("*"))[:20]:
        if p.is_file():
            print(" ", p.relative_to(coord_root))
